# GymLeader: SFT of Qwen3.5-2B on Google Colab

LoRA fine-tuning of Qwen3.5-2B on our workout dataset, following Unsloth's guide
(https://unsloth.ai/docs/models/qwen3.5/fine-tune). The result goes to the Hugging Face Hub, and the Mac evaluates it from there.

## How to run
1. Runtime → Change runtime type → **T4 GPU** or better (L4, A100).
2. Add the Colab secret `HF_TOKEN` (key icon on the left): a token with **write** access from
   https://huggingface.co/settings/tokens, with **notebook access** switched on. Your username is read from the token.
3. Run the install cell, then **Runtime → Restart session**.
4. Click the second cell and choose **Runtime → Run cell and below**. Each section asserts its own result, so a problem
   stops the run where it happens.

Nothing is saved during training (about 14 minutes): after a disconnect, start again from step 3.
`test.jsonl` is never loaded: the test set is only for the final evaluation on the Mac.

## 1. Install

Qwen3.5 needs transformers v5 and Colab ships v4. We follow Unsloth's forced, no-cache reinstall and ask for v5 in the same
command, so pip picks versions that work together or fails loudly. Restart the session afterwards.

**Check:** pip ends without errors.

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo "transformers>=5,<6"

## 2. Versions, settings, GPU and Hugging Face login

Everything that can fail does so here, before the slow steps: the transformers version, the GPU and the Hugging Face token
(it must have write access; `HF_USER`, the owner of the repos in section 13, is read from it).
All settings live in this cell: 16 rows per step × 125 steps per epoch × 2 epochs = 250 steps.
Outputs go to `RUN_DIR` on Colab's disk, which is deleted when the session ends: the Hub upload (section 13) is what we keep.

**Check:** transformers 5.x, 250 total steps, the GPU name and memory, "logged in to Hugging Face as <HF_USER>".

In [ ]:
import unsloth  # import first: Unsloth patches transformers/trl when it is imported
import math, torch, transformers, trl
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

for lib in (unsloth, transformers, trl, torch):
    print(f"{lib.__name__:13} {lib.__version__}")
assert transformers.__version__.startswith("5."), "Qwen3.5 needs transformers v5: restart the session and re-run"

MODEL_NAME = "Qwen/Qwen3.5-2B"
MAX_SEQ_LENGTH = 1024
LORA_R, LORA_ALPHA = 16, 16
LEARNING_RATE = 2e-4
BATCH_SIZE = 8           # rows per GPU step
GRAD_ACCUM = 2           # GPU steps per optimizer step
EPOCHS = 2
WARMUP_STEPS = 10
EVAL_STEPS, LOGGING_STEPS = 25, 5
SEED = 42
RUN_NAME = "sft_lora_r16"
RUN_DIR = Path("/content/run") / RUN_NAME

TOTAL_STEPS = math.ceil(2000 / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nrun {RUN_NAME} -> {RUN_DIR}, {TOTAL_STEPS} total steps")

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.is_bf16_supported()
print(f"GPU {gpu.name}, {gpu.total_memory / 1e9:.1f} GB, bf16 {BF16}")
if "T4" in gpu.name:
    print("Note: T4 detected. Qwen3.5's kernels compile slowly on T4, so the first steps will be slow. That is expected.")

api = HfApi(token=userdata.get("HF_TOKEN"))  # fails if the secret is missing or notebook access is off
me = api.whoami()
HF_USER = me["name"]  # the repos are created under this account
role = me.get("auth", {}).get("accessToken", {}).get("role")
assert role != "read", "HF_TOKEN is read-only: section 13 needs a token with write access"
print(f"logged in to Hugging Face as {HF_USER} (token role: {role})")

## 3. Data

We clone `main` for today's code, but the training data must be the frozen dataset tagged `data-v1`, so we assert that
`data/processed/` is identical to the tag. Only train and val are loaded: looking at the test set during training would make
the test score meaningless.

**Check:** "data/processed/ matches data-v1" and "train 2000, val 200".

In [ ]:
import json, subprocess, sys

REPO = Path("/content/SFT-Gym-Workout-Program")
if not REPO.exists():
    !git clone --quiet --branch main https://github.com/riccardoroselli/SFT-Gym-Workout-Program.git {REPO}
!git -C {REPO} fetch --quiet --tags
!git -C {REPO} log -1 --oneline
diff = subprocess.run(["git", "-C", str(REPO), "diff", "--quiet", "data-v1", "HEAD", "--", "data/processed/"])
assert diff.returncode == 0, "data/processed/ on main differs from data-v1: the training data is not the frozen dataset"
print("data/processed/ matches data-v1")
sys.path.insert(0, str(REPO / "src"))  # so we can import evaluate.py and verifier.py

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]

train = read_jsonl(REPO / "data/processed/train.jsonl")
val = read_jsonl(REPO / "data/processed/val.jsonl")
assert (len(train), len(val)) == (2000, 200), (len(train), len(val))
print(f"train {len(train)}, val {len(val)}")

## 4. Model

16-bit LoRA as in Unsloth's guide (4-bit QLoRA loses more accuracy on Qwen3.5). We count the parameters per dtype to see what
the weights really are: a T4 has no bf16, so Unsloth keeps them in float32. The end-of-turn token must be `<|im_end|>`,
because that's how the model learns to stop after the JSON.

**Check:** almost all parameters in one dtype (bfloat16 on L4/A100, float32 on a T4); eos_token `<|im_end|>`.

In [ ]:
import collections
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated, "
      f"{torch.cuda.memory_reserved() / 1e9:.2f} GB reserved")

by_dtype = collections.Counter()
for p in model.parameters():
    by_dtype[str(p.dtype)] += p.numel()
for dtype, n in by_dtype.most_common():
    print(f"  {dtype:15} {n:>14,} parameters")

END = "<|im_end|>"
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)  # Qwen3.5 is a vision-language model: processor -> text tokenizer
print("eos_token:", repr(text_tokenizer.eos_token), "id", text_tokenizer.eos_token_id)
assert text_tokenizer.eos_token == END

## 5. Chat template check

The model must train on exactly the text it will see at evaluation, so we import `build_prompt()` from `src/evaluate.py`
instead of copying it (mlx is imported only inside its `generate` step, so this works on Colab). A training row must be
that prompt, then the answer JSON, then `<|im_end|>`.

**Check:** "template check passed", and the longest train row is about 209 tokens.

In [ ]:
from transformers import AutoTokenizer
from evaluate import build_prompt

reference = AutoTokenizer.from_pretrained(MODEL_NAME)

def train_text(row):
    return text_tokenizer.apply_chat_template(row["messages"], tokenize=False, enable_thinking=False)

system, user, assistant = train[0]["messages"]
text = train_text(train[0])
print(text)
eval_prompt = build_prompt(reference, [system, user])  # what evaluate.py sends, with no few-shot turns
assert text.startswith(eval_prompt), "training text does not start with the evaluate.py prompt"
assert text.rstrip("\n").endswith(assistant["content"] + END), "training text does not end with answer + <|im_end|>"
assert text == eval_prompt + assistant["content"] + END + "\n", "unexpected extra text"
print("template check passed: training text = evaluate.py prompt + answer JSON + <|im_end|>")

lengths = [len(text_tokenizer(train_text(r))["input_ids"]) for r in train]
longest = max(range(len(train)), key=lengths.__getitem__)
print(f"longest train row: {lengths[longest]} tokens (row {longest}), max_seq_length {MAX_SEQ_LENGTH}")
assert lengths[longest] <= MAX_SEQ_LENGTH

## 6. LoRA

LoRA freezes the 2B base weights and trains small extra matrices (rank 16) next to the attention and MLP projections:
far less memory, and a small adapter file. The settings follow Unsloth's guide.

**Check:** the trainable parameters are around 1% of the total or less.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} of {total:,} parameters ({100 * trainable / total:.2f}%)")

## 7. Datasets, trainer and masking check

Each row becomes a prompt-completion pair, and `completion_only_loss=True` puts the loss on the answer only: the model
learns to write the answer, not to repeat the request. The masking check confirms it on one tokenized row.
No checkpoints: training is short, so after a disconnect we run it again. We print the weights' real dtype, because on a
T4 Unsloth trains in float32 whatever the fp16 flag says.

**Check:** the asserts pass; weights bfloat16 (L4/A100) or float32 (T4); "masking check passed".

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def prompt_completion(rows):
    return Dataset.from_list([{"prompt": build_prompt(text_tokenizer, r["messages"][:2]),
                               "completion": r["messages"][2]["content"] + END} for r in rows])

train_ds, val_ds = prompt_completion(train), prompt_completion(val)
assert all(p == build_prompt(reference, r["messages"][:2]) for p, r in zip(train_ds["prompt"], train))
print(train_ds, val_ds, sep="\n")

trainer = SFTTrainer(model=model, processing_class=text_tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
                     args=SFTConfig(
    output_dir=str(RUN_DIR / "trainer"), max_length=MAX_SEQ_LENGTH, completion_only_loss=True, dataset_num_proc=1,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_scheduler_type="cosine",
    optim="adamw_8bit", weight_decay=0.01, seed=SEED,
    eval_strategy="steps", eval_steps=EVAL_STEPS, logging_steps=LOGGING_STEPS, save_strategy="no",
    bf16=BF16, fp16=not BF16, report_to="none",
))
lora_dtypes = {str(p.dtype) for p in model.parameters() if p.requires_grad}
base_dtypes = {str(p.dtype) for p in model.parameters() if not p.requires_grad}
print(f"trained (LoRA) weights: {lora_dtypes}, frozen base weights: {base_dtypes}")  # the real precision
print(f"flags passed: bf16={trainer.args.bf16}, fp16={trainer.args.fp16}")

batch = trainer.data_collator([trainer.train_dataset[0]])  # the tokenized version of train_ds[0]
ids, labels = batch["input_ids"][0], batch["labels"][0]
in_loss = ids[labels != -100]                               # -100 = ignored by the loss
print(f"{len(in_loss)} of {len(ids)} tokens count in the loss:", repr(text_tokenizer.decode(in_loss)))
assert text_tokenizer.decode(ids) == train_ds[0]["prompt"] + train_ds[0]["completion"]
assert text_tokenizer.decode(in_loss) == train_ds[0]["completion"], "loss is not on exactly the answer + <|im_end|>"
print("masking check passed")

## 8. Train

2 epochs = 250 steps, with a val eval every 25. The log history is saved, so the curves can be redrawn and uploaded with
the model.

**Check:** step 250 in about 14 minutes on a T4; `log_history.json` saved.

In [ ]:
import time

torch.cuda.reset_peak_memory_stats()
start = time.time()
trainer.train()
print(f"total time {(time.time() - start) / 60:.1f} min, peak GPU memory {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
(RUN_DIR / "log_history.json").write_text(json.dumps(trainer.state.log_history, indent=1))
print("saved", RUN_DIR / "log_history.json")

## 9. Loss curves

Train loss is on the batches the model learns from, val loss on 200 rows it never sees. Both falling and ending close
means it learns things that carry over; val rising while train falls means overfitting.

**Check:** both curves go down and end close together.

In [ ]:
import matplotlib.pyplot as plt

history = json.loads((RUN_DIR / "log_history.json").read_text())
train_pts = [(e["step"], e["loss"]) for e in history if "loss" in e]
val_pts = [(e["step"], e["eval_loss"]) for e in history if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(7, 4))
for (pts, label, color, marker) in [(train_pts, "train loss", "#2a78d6", None), (val_pts, "val loss", "#eb6834", "o")]:
    ax.plot(*zip(*pts), label=f"{label} (final {pts[-1][1]:.4f})", color=color, linewidth=2, marker=marker, markersize=6)
ax.set(xlabel="step", ylabel="loss", title=f"{RUN_NAME}: train and val loss")
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.savefig(RUN_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"final train loss {train_pts[-1][1]:.4f} (step {train_pts[-1][0]}), "
      f"final val loss {val_pts[-1][1]:.4f} (step {val_pts[-1][0]})")

## 10. Quick check on val (not test)

A sanity check, not the real evaluation (that is the test set, on the Mac): 30 val rows with the same prompt, greedy
decoding and `verify()` as `evaluate.py`. The answers are saved for the reload test.

**Check:** parse_ok and strict_format near 1.0, names_exist and all_correct far above the baseline (0.16 and 0.0), truncated 0.

In [ ]:
from evaluate import FIELDS, MAX_NEW_TOKENS, mean
from verifier import verify

FastLanguageModel.for_inference(model)
IM_END_ID = text_tokenizer.convert_tokens_to_ids(END)

def answer(model, row):
    "Greedy answer to a row's request, like evaluate.py. -> (text, truncated)"
    prompt = build_prompt(text_tokenizer, row["messages"][:2])
    inputs = text_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         eos_token_id=IM_END_ID, pad_token_id=text_tokenizer.pad_token_id)
    new = out[0, inputs["input_ids"].shape[1]:]
    truncated = len(new) == MAX_NEW_TOKENS and new[-1].item() != IM_END_ID
    return text_tokenizer.decode(new, skip_special_tokens=True), truncated

results = []
for i, row in enumerate(val[:30], 1):
    text, truncated = answer(model, row)
    results.append({"output": text, **verify(row["meta"], text), "has_think": "<think>" in text, "truncated": truncated})
    print(f"[{i}/30] reward {results[-1]['reward']:.2f}  {text[:70]!r}")
(RUN_DIR / "val_check.json").write_text(json.dumps(results, indent=1))

print()
for k in FIELDS:
    print(f"{k:14} {mean(r[k] for r in results)}")
for row, r in list(zip(val, results))[:3]:
    print("\nUSER:  ", row["messages"][1]["content"], "\nMODEL: ", r["output"])

## 11. Save the adapter and the merged model

The adapter is small; the merged model (LoRA added into the base) is what the Mac evaluates. The Mac baseline runs in bf16,
so any tensor the merge wrote in another dtype is cast back to the base's dtype: otherwise the SFT result would mix training
with a precision change. Finally, the saved tokenizer must still build exactly the `evaluate.py` prompt.

**Check:** shards only BF16 (plus a few F32), torch_dtype bfloat16, "tokenizer check passed".

In [ ]:
ADAPTER_DIR, MERGED_DIR = RUN_DIR / "adapter", RUN_DIR / "merged"
model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from safetensors.torch import load_file, save_file

TORCH_DTYPE = {"BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32}

def header_dtypes(path):
    with safe_open(str(path), "pt") as f:
        return {k: f.get_slice(k).get_dtype() for k in f.keys()}

base = header_dtypes(hf_hub_download(MODEL_NAME, "model.safetensors"))  # in the HF cache since section 4
for shard in sorted(MERGED_DIR.glob("*.safetensors")):
    saved = header_dtypes(shard)
    wrong = {k for k, d in saved.items() if d in TORCH_DTYPE and d != base.get(k, "BF16")}
    if wrong:
        tensors = load_file(str(shard))
        for k in wrong:
            tensors[k] = tensors[k].to(TORCH_DTYPE[base.get(k, "BF16")])
        save_file(tensors, str(shard), metadata={"format": "pt"})
    print(f"{shard.name}: {dict(collections.Counter(header_dtypes(shard).values()))}, cast {len(wrong)} tensors, "
          f"{len(saved.keys() & base.keys())}/{len(saved)} names found in the base")

config_path = MERGED_DIR / "config.json"
config = json.loads(config_path.read_text())
for c in (config, config.get("text_config", {})):
    for key in ("torch_dtype", "dtype"):
        if key in c:
            c[key] = "bfloat16"
config_path.write_text(json.dumps(config, indent=2))

In [ ]:
for folder in (ADAPTER_DIR, MERGED_DIR):
    print(folder)
    for f in sorted(folder.iterdir()):
        print(f"  {f.name:40} {f.stat().st_size / 1e6:10.1f} MB")
    saved_tok = AutoTokenizer.from_pretrained(str(folder))
    assert all(build_prompt(saved_tok, r["messages"][:2]) == build_prompt(reference, r["messages"][:2]) for r in val), folder

config = json.loads((MERGED_DIR / "config.json").read_text())
text_config = config.get("text_config", {})
print("architectures:", config.get("architectures"))
print("torch_dtype:  ", config.get("torch_dtype", config.get("dtype")),
      "| text_config:", text_config.get("torch_dtype", text_config.get("dtype")))
print("tokenizer check passed: the saved chat templates give the same build_prompt output")

## 12. Reload test

We load the merged model back, as the Mac will, and answer 3 val rows again: with greedy decoding they should match
section 10. If an answer no longer parses or loses more than 0.1 reward, the run stops here, before the broken model
is uploaded. The training model is freed first, so the two don't share the GPU.

**Check:** 3 × MATCH, and no assert error. A DIFF with about the same reward is a tiny rounding difference (merged bf16 vs
base + LoRA).

In [ ]:
import gc

for name in ("trainer", "model"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

reloaded, _ = FastLanguageModel.from_pretrained(str(MERGED_DIR), max_seq_length=MAX_SEQ_LENGTH,
                                                load_in_4bit=False, load_in_16bit=True)
FastLanguageModel.for_inference(reloaded)
before = json.loads((RUN_DIR / "val_check.json").read_text())
for row, old in zip(val[:3], before[:3]):
    new, _ = answer(reloaded, row)
    check = verify(row["meta"], new)
    print("MATCH" if new == old["output"] else "DIFF ", f"reward {check['reward']:.2f} (section 10: {old['reward']:.2f})", repr(new[:80]))
    assert check["parse_ok"] and check["reward"] >= old["reward"] - 0.1, "the reloaded merged model answers worse: not uploading it"

## 13. Upload to the Hugging Face Hub

Colab's disk is temporary, so the model lives on the Hub: each upload is a versioned commit that names the code commit, the
Mac downloads it straight into `HF_HOME`, and the model cards make a portfolio page. `upload_folder` (not
`push_to_hub_merged`, which would merge again) uploads exactly the files we checked; the merged repo also gets the log, the
chart and the val check.

**Check:** two private repo URLs printed; each model page shows its card.

In [ ]:
import shutil

GITHUB = "https://github.com/riccardoroselli/SFT-Gym-Workout-Program"
CODE_COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()
ADAPTER_REPO, MERGED_REPO = f"{HF_USER}/qwen3.5-2b-gym-sft-lora", f"{HF_USER}/qwen3.5-2b-gym-sft"

history = json.loads((RUN_DIR / "log_history.json").read_text())
final_train = [e["loss"] for e in history if "loss" in e][-1]
final_val = [e["eval_loss"] for e in history if "eval_loss" in e][-1]
val_check = json.loads((RUN_DIR / "val_check.json").read_text())
means = {k: mean(r[k] for r in val_check) for k in FIELDS}

def model_card(kind):
    if kind == "merged":
        usage = f"```bash\npython src/evaluate.py generate --run {RUN_NAME} --model {MERGED_REPO}\n```"
        chart = f"![train and val loss](https://huggingface.co/{MERGED_REPO}/resolve/main/loss_curves.png)\n\n"
    else:
        usage = f"Load it with PEFT on top of `{MODEL_NAME}`, or use the merged model: [{MERGED_REPO}](https://huggingface.co/{MERGED_REPO})."
        chart = ""
    rows = "\n".join(f"| {k} | {'-' if v is None else v} |" for k, v in means.items())
    return f'''---
base_model: {MODEL_NAME}
library_name: {"peft" if kind == "adapter" else "transformers"}
tags: [sft, lora, unsloth, trl, json, fitness]
---

# {ADAPTER_REPO if kind == "adapter" else MERGED_REPO}

{MODEL_NAME} fine-tuned with SFT (LoRA) to write gym workouts as JSON from requests like
"chest workout, medium difficulty, free weights". It answers `{{"workout": [{{"name", "sets", "reps"}}, ...]}}` with exercise
names from the [exercise catalog](https://github.com/hasaneyldrm/exercises-dataset), or `{{"error": ...}}` when the request
is impossible. This repo holds the **{"LoRA adapter" if kind == "adapter" else "merged 16-bit model (base + LoRA, bf16)"}**.

**This is an SFT checkpoint, not the final model:** the next stage is GRPO.

- Code: [{GITHUB}]({GITHUB}), commit [`{CODE_COMMIT[:7]}`]({GITHUB}/commit/{CODE_COMMIT})
- Data: tag [`data-v1`]({GITHUB}/tree/data-v1/data/processed) (2,000 train rows, 200 val rows; the test set was not used here)
- Prompt: the dataset's system message + the request, Qwen chat template with thinking off

## Training

| setting | value |
|---|---|
| method | bf16 LoRA (not QLoRA) with Unsloth + TRL `SFTTrainer`, loss on the answer only |
| LoRA | r={LORA_R}, alpha={LORA_ALPHA}, dropout 0, on q/k/v/o/gate/up/down projections |
| learning rate | {LEARNING_RATE}, cosine, {WARMUP_STEPS} warmup steps, 8-bit AdamW, weight decay 0.01 |
| batch | {BATCH_SIZE} × {GRAD_ACCUM} gradient accumulation = {BATCH_SIZE * GRAD_ACCUM} rows per step |
| epochs / steps | {EPOCHS} / {TOTAL_STEPS} |
| max length / seed | {MAX_SEQ_LENGTH} / {SEED} |
| GPU | {torch.cuda.get_device_name(0)} |

{chart}Final train loss **{final_train:.4f}**, final val loss **{final_val:.4f}**.

## Quick check

{len(val_check)} val rows (not test), greedy decoding, max {MAX_NEW_TOKENS} new tokens, scored with the project's `verify()`.
The real evaluation, on the 300-row test set, runs separately on a Mac with MLX.

| check | mean |
|---|---|
{rows}

## Known weakness

**Duplicates:** the model sometimes lists the same exercise twice in one workout (`no_duplicates` = {means["no_duplicates"]}
on the check above). The next stage (GRPO, with the verifier as reward) is meant to fix this.

## Usage

{usage}
'''

for name in ("log_history.json", "loss_curves.png", "val_check.json"):
    shutil.copy(RUN_DIR / name, MERGED_DIR / name)  # uploaded with the merged model
for kind, repo, folder in [("adapter", ADAPTER_REPO, ADAPTER_DIR), ("merged", MERGED_REPO, MERGED_DIR)]:
    api.create_repo(repo, private=True, exist_ok=True)
    (folder / "README.md").write_text(model_card(kind))  # replaces the auto-generated card
    api.upload_folder(repo_id=repo, folder_path=str(folder),
                      commit_message=f"{RUN_NAME} {kind}, trained with {GITHUB}/commit/{CODE_COMMIT}")
    print(f"{kind:8} -> https://huggingface.co/{repo}")
print(f"\nOn the Mac: python src/evaluate.py generate --run {RUN_NAME} --model {MERGED_REPO}")

## 14. Next steps (on the Mac)

The repos are private, so the Mac reads the token from `.env` (not committed). Replace `<HF_USER>` with yours; `mlx_lm.load()`
downloads the model into `HF_HOME` the first time.
```bash
conda activate GymLeaderProject
cd "/Volumes/lexar/produttività/projects/SFT-Gym-Workout-Program"
set -a; source .env; set +a    # puts HF_TOKEN in this terminal's environment
python src/evaluate.py generate --run sft_lora_r16 --model <HF_USER>/qwen3.5-2b-gym-sft
python src/evaluate.py score --run sft_lora_r16
python src/analyze_errors.py --run sft_lora_r16
```

**Check:** `eval/results/results.md` gets an `sft_lora_r16` row, next to the two baselines.